# Musanga V4 — NB06 v4_004 | Scientific report and real figure atlas

**Deliverable:** an illustrated, editable **Word (.docx) scientific progress report**, all publication-quality PNG figures, consolidated summary CSV tables, and a reproducibility manifest, saved to `Final/Outputs/v4/06_scientific_report_v1/`. Generated **directly from actual reference geometries, RGB/DSM/MS rasters and cached NB00/NB04/NB05 metrics**—no invented example images and no neural-network inference or training.

**Report sections:** objectives and hypotheses; study sites and reference datasets; annotation examples; Nadine's exhaustive UAV crown fractions vs independent field Musanga basal-area fraction; RGB/DSM/MS examples for actual matching reference footprints; processing and spatial evaluation; 106-footprint RGB baseline; 92-footprint sensor-matched comparison; site-specific performance, high-cover MOS and uncertainty; limitations and next experiments.

**Critical R² clarification:** MOS RGB R² = **0.164** on **all 29** NB04 MOS reference footprints with full RGB support. MOS RGB R² = **−0.025** on the **23** NB05 multisensor-matched MOS footprints, evaluated on a *different common-valid pixel support*. Both are correct; the change is not evidence of a changed RGB model. Historical 0.186 was an alternative weighting (50/50 circle vs plot). NB06 calculates and labels these different estimands separately.

**How to run:** Google Colab **CPU is sufficient**; execute cells in order. The figure panels use bounded, downsampled raster reads to avoid large orthomosaic loads. If the combined RGB+DSM+MS (fourth) model has finished, NB06 automatically includes it; otherwise the three completed arms are reported and the fourth is marked pending. The report does **not** claim model-generated segmentation mask overlays unless such pixel probability rasters actually exist.

**Source caution:** plot-background exhaustiveness is inherited from the V4 reference contract. Comparing field percentage of Musanga basal area with annotated UAV crown-area fraction is an *ecological cross-variable comparison*, not direct agreement between measurements of the same quantity. Site/fold cross-validation is spatial-unit holdout, not whole-site leave-one-site-out.

## 00 — Mount Drive, read-only source files and output folder

In [ ]:
%pip -q install geopandas pyogrio shapely rasterio matplotlib python-docx scipy
from pathlib import Path
import os,json,math,shutil,warnings,hashlib
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window,from_bounds
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.transform import Affine
from rasterio.features import rasterize
from shapely import make_valid
from shapely.geometry import box
from shapely.ops import unary_union
from contextlib import ExitStack
from IPython.display import display,Image
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=next((p for p in (
    Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
    Path('/content/drive/MyDrive/Musanga/Final')) if p.exists()),None)
assert ROOT is not None,'Final data root missing'
V4=ROOT/'Outputs/v4'
INPUT00=V4/'00_input_audit'
EVAL4=V4/'04_paired_rgb_heldout_v1'
NB05=V4/'05_rgb_dsm_ms_matched_v1'
OUT=V4/'06_scientific_report_v1'
FIG=OUT/'figures'
FIG.mkdir(parents=True,exist_ok=True)
F78=json.loads((INPUT00/'00_v4_field78_contract.json').read_text())
V31=json.loads((ROOT/'Outputs/V3_1/00_input_audit/00_input_manifest.json').read_text())
SPLIT_ROOT=V4/'01_spatial_cover_folds_v2'
NB02=V4/'02_rgb_circles_plus_plots_v3'
OWNERS=pd.read_csv(NB02/'02_reference_mosaic_owners_v2.csv')
OWNERS.unit_id=OWNERS.unit_id.astype(str)
BASE_CSV=EVAL4/'04_heldout_unit_metrics.csv'
assert BASE_CSV.exists(),BASE_CSV
MOSAICS={}
for rec in V31['mosaics']:
    name=str(rec['file'])
    if name.lower().endswith('.tif'):
        p=Path(rec['path'])
        if not p.exists():p=ROOT/'Data/Ortho_RGB'/name
        if p.exists():MOSAICS[name]=p
assert len(MOSAICS)==6,MOSAICS
print('Output:',OUT)
print('No GPU, inference or retraining. Existing model results are read-only.')

## 01 — Load and audit NB04, NB05 and independent Nadine field-comparison statistics

In [ ]:
RGB=pd.read_csv(BASE_CSV)
assert 'arm' in RGB
RGB=RGB.loc[RGB.arm.eq('A')].copy()
RGB.unit_id=RGB.unit_id.astype(str)
assert len(RGB)==106 and RGB[['unit_type','unit_id']].duplicated().sum()==0
RGB['cover_error_pp']=RGB.predicted_cover_pp-RGB.observed_cover_pp
RGB['abs_cover_error_pp']=RGB.cover_error_pp.abs()

METRIC_PATH=NB05/'06_all_three_models_matched_unit_metrics.csv'
FOUR_PATH=NB05/'08_four_arm_equal_footprint_unit_metrics.csv'
assert METRIC_PATH.exists(),f'Run NB05 section 06 first: {METRIC_PATH}'
MATCH=pd.read_csv(FOUR_PATH if FOUR_PATH.exists() else METRIC_PATH)
MATCH.unit_id=MATCH.unit_id.astype(str)
models=sorted(MATCH.model.unique())
expected={'RGB','RGB+DSM','RGB+MS'}
assert expected.issubset(set(models)),models
nmatch=MATCH.groupby(['unit_type','unit_id']).ngroups
assert nmatch==92,(nmatch,'Historical matched-unit count differs')
assert MATCH.groupby(['unit_type','unit_id']).model.nunique().eq(len(models)).all()
assert MATCH.groupby(['unit_type','unit_id']).n_pixels.nunique().eq(1).all()
assert MATCH.groupby(['unit_type','unit_id']).observed_cover_pp.apply(
    lambda x:np.ptp(x.to_numpy(float))<1e-6).all()
if 'RGB+DSM+MS' not in models:
    print('Fourth model not yet found — three completed arms reported; combination marked pending.')
else:
    print('Fourth model completed — including 17-channel combination in report.')

NADINE_CSV=INPUT00/'00_v4_field_BA_vs_Nadine_78plots.csv'
NADINE_STATS=INPUT00/'00_v4_field_BA_vs_Nadine_78plot_statistics.csv'
assert NADINE_CSV.exists() and NADINE_STATS.exists(),(
    'NB00 section 07 field comparison was not saved')
NADINE=pd.read_csv(NADINE_CSV)
NADINE_SUMMARY=pd.read_csv(NADINE_STATS)
assert len(NADINE)==78
print('106 full RGB footprints | 92 sensor-matched footprints | 78 Nadine field plots')
print('NB00 independent field BA validation primary subset:',
      int(NADINE.primary_BA_validation.sum()),'plots')

def unit_metrics(frame):
    x=frame.observed_cover_pp.to_numpy(float)
    y=frame.predicted_cover_pp.to_numpy(float)
    diff=y-x
    variance=float(np.mean((x-x.mean())**2))
    rmse=float(np.sqrt(np.mean(diff*diff)))
    sd=float(np.std(x,ddof=1)) if len(x)>1 else np.nan
    return dict(n=len(frame),R2=1-rmse*rmse/variance if variance>0 else np.nan,
        MAE_pp=float(np.mean(np.abs(diff))),RMSE_pp=rmse,Bias_pp=float(np.mean(diff)),
        F1=float(frame.F1.mean()),IoU=float(frame.IoU.mean()),
        Precision=float(frame.precision.mean()),Recall=float(frame.recall.mean()),
        RPD=sd/rmse if rmse>0 else np.nan,
        RPIQ=float(np.subtract(*np.percentile(x,[75,25])))/rmse if rmse>0 else np.nan,
        Brier=float(frame.brier.mean()))
ALL_RGB=pd.DataFrame([dict(Site=site,**unit_metrics(g)) for site,g in
    [('ALL',RGB),*list(RGB.groupby('site'))]])
MATCH_SUM=pd.DataFrame([dict(Site=site,Model=model,**unit_metrics(g))
    for site,s in [('ALL',MATCH),*list(MATCH.groupby('site'))]
    for model,g in s.groupby('model')])
ALL_RGB.to_csv(OUT/'table_01_rgb_full_106.csv',index=False)
MATCH_SUM.to_csv(OUT/'table_02_matched_model_comparison.csv',index=False)
MOS_COMPARE=pd.DataFrame([
    dict(Evaluation='NB04 full RGB footprints',**unit_metrics(RGB[RGB.site=='MOS'])),
    dict(Evaluation='NB05 matched RGB footprints',
         **unit_metrics(MATCH[(MATCH.site=='MOS')&(MATCH.model=='RGB')]))
])
assert len(RGB[RGB.site=='MOS'])==29
assert len(MATCH[(MATCH.site=='MOS')&(MATCH.model=='RGB')])==23
MOS_COMPARE.to_csv(OUT/'table_03_MOS_R2_reconciliation.csv',index=False)
print('RGB complete 106-footprint performance:')
display(ALL_RGB.round(3))
print('Strictly matched modality evaluation:')
display(MATCH_SUM.round(3))
print('MOS R² discrepancy explained by support differences:')
display(MOS_COMPARE[['Evaluation','n','R2','MAE_pp','RMSE_pp']].round(3))


## 01B — Key results and source coverage: summary tables for colleagues

The tables below are generated from saved NB04/NB05 scores, not from manually pasted numbers. The **primary comparison gives equal weight to each reference footprint**. A second, clearly identified pixel-weighted summary is supplementary and should not be conflated with the primary estimand.

The first table accounts for all 106 footprints, including the 14 without complete multisensor support. The second is the full 106-footprint RGB baseline, and the third is the **92-footprint strictly matched** modality comparison (by forest and overall). A compact summary places the distinct denominators side by side. If the fourth arm has finished, it is included automatically.

In [ ]:
# 01B — Scientific results-at-a-glance (read existing cached NB04/NB05 data).
# All tables are written to disk and embedded at the front of the Word report.
from IPython.display import display,Markdown
SUMMARY_COLS=['Site','Model','n','F1','IoU','Precision','Recall','R2',
              'MAE_pp','RMSE_pp','Bias_pp','RPD','RPIQ','Brier']
MODEL_ORDER=['RGB','RGB+DSM','RGB+MS','RGB+DSM+MS']
SITE_ORDER=['ALL','Baego','MOS','Yangambi','Yoko']
SUPPORT_FILE=NB05/'06_all_106_unit_support.csv'
assert SUPPORT_FILE.is_file(),('Run NB05 reporting section first',SUPPORT_FILE)
SUP=pd.read_csv(SUPPORT_FILE)
assert len(SUP)==106 and {'site','unit_type','n_common'}.issubset(SUP.columns)
SUP['matched']=SUP.n_common.gt(0)
SUPPORT_TABLE=(SUP.groupby(['site','unit_type'],sort=False)
               .agg(total=('matched','size'),
                    matched=('matched','sum'),
                    common_pixels=('n_common','sum')).reset_index())
SUPPORT_TABLE['missing']=SUPPORT_TABLE.total-SUPPORT_TABLE.matched
assert int(SUPPORT_TABLE.total.sum())==106
assert int(SUPPORT_TABLE.matched.sum())==92
assert int(SUPPORT_TABLE.missing.sum())==14
SUPPORT_TABLE.to_csv(OUT/'table_00_multisensor_support_by_site_and_source.csv',index=False)

# Source-faithful reporting: reuse NB05 precomputed results when available,
# rather than silently substituting a different dispersion/quantile convention.
REPORT_CSV=NB05/'06_weighted_model_comparison.csv'
if REPORT_CSV.is_file() and 'RGB+DSM+MS' not in models:
    exact=pd.read_csv(REPORT_CSV)
    exact=exact.loc[exact.Weighting.eq('Equal units')].copy()
    exact=exact.rename(columns={'Group':'Site','model':'Model'})
    SUMMARY_MAIN=exact[SUMMARY_COLS].copy()
else:
    SUMMARY_MAIN=MATCH_SUM[SUMMARY_COLS].copy()
SUMMARY_MAIN['Site']=pd.Categorical(SUMMARY_MAIN.Site,categories=SITE_ORDER,ordered=True)
SUMMARY_MAIN['Model']=pd.Categorical(SUMMARY_MAIN.Model,categories=MODEL_ORDER,ordered=True)
SUMMARY_MAIN=SUMMARY_MAIN.sort_values(['Site','Model']).reset_index(drop=True)
SUMMARY_MAIN['Site']=SUMMARY_MAIN.Site.astype(str)
SUMMARY_MAIN['Model']=SUMMARY_MAIN.Model.astype(str)
SUMMARY_MAIN.to_csv(OUT/'table_00_primary_equal_footprint_performance.csv',index=False)

# Clearly distinguish the complete full-RGB assessment from the 92-sample comparison.
FULL_RGB=ALL_RGB.copy()
FULL_RGB['Model']='RGB (NB04; complete footprint)'
FULL_RGB=FULL_RGB[SUMMARY_COLS].copy()
FULL_RGB.to_csv(OUT/'table_00_complete_RGB_106.csv',index=False)

AGG=SUMMARY_MAIN[SUMMARY_MAIN.Site.eq('ALL')].copy()
COMPACT=pd.concat([FULL_RGB[FULL_RGB.Site.eq('ALL')],AGG],ignore_index=True)
COMPACT.to_csv(OUT/'table_00_key_results_compact.csv',index=False)

print('Support:',int(SUPPORT_TABLE.total.sum()),'RGB footprints;',
      int(SUPPORT_TABLE.matched.sum()),'matched;',int(SUPPORT_TABLE.missing.sum()),'unmatched')
print('MAIN MODEL COMPARISON — 92 matched footprints')
display(SUMMARY_MAIN[SUMMARY_MAIN.Site.eq('ALL')][
    ['Model','n','R2','MAE_pp','RMSE_pp','Bias_pp','F1','IoU']].round(3))
print('MOS R² changes with evaluation support, not model weights:')
display(MOS_COMPARE[['Evaluation','n','R2']].round(3))
# A supplementary area/pixel-weighted view is explicitly NOT the primary metric.
PIXEL_TABLE=None
if REPORT_CSV.is_file():
    x=pd.read_csv(REPORT_CSV)
    x=x[(x.Group=='ALL')&(x.Weighting=='Valid-pixel weighted')]
    if len(x):
        PIXEL_TABLE=x.rename(columns={'model':'Model'})[
            ['Model','Weighting','n','F1','IoU','Precision','Recall','R2',
             'MAE_pp','RMSE_pp','Bias_pp','RPD','RPIQ','Brier']].copy()
        PIXEL_TABLE.to_csv(OUT/'table_00_supplementary_pixel_weighted.csv',index=False)
        print('Supplementary pixel-weighted metrics saved to CSV (not shown in concise report).')
print('Saved scientific summary tables in',OUT)


## 01C — Global performance by Musanga-cover class (best model: RGB + MS)

One compact table for **RGB+MS only**, across the **92 sensor-matched reference footprints**. Classes follow NB05: 0–1%, 1–5%, 5–20%, 20–40% and >40%. Each footprint is weighted equally. In narrow cover classes, R² can be misleading because the observed variance is small, so the report emphasizes MAE, RMSE, bias and F1. No site-by-class tables or alternate-model class tables are included.

In [ ]:
# 01C — Compact cover-class performance: best operational candidate RGB+MS.
BINS=[-.001,1,5,20,40,100.001]
CLASS_LABELS=['0–1%','1–5%','5–20%','20–40%','>40%']
BEST_MODEL='RGB+MS'
best=MATCH[MATCH.model.eq(BEST_MODEL)].copy()
assert len(best)==92 and best[['unit_type','unit_id']].duplicated().sum()==0
best['Cover class']=pd.cut(best.observed_cover_pp,bins=BINS,
                          labels=CLASS_LABELS,include_lowest=True)
assert best['Cover class'].notna().all()
strata=[]
for label,part in best.groupby('Cover class',observed=True):
    q=unit_metrics(part)
    # A restricted observed-cover range often destabilizes R²; omit from
    # publication-facing class table, while keeping full main model metrics.
    strata.append({'Cover class':str(label),'n':len(part),
                   'F1':q['F1'],'IoU':q['IoU'],
                   'MAE (pp)':q['MAE_pp'],'RMSE (pp)':q['RMSE_pp'],
                   'Bias (pp)':q['Bias_pp']})
CLASS_BEST=pd.DataFrame(strata)
CLASS_BEST['Cover class']=pd.Categorical(CLASS_BEST['Cover class'],
                                          categories=CLASS_LABELS,ordered=True)
CLASS_BEST=CLASS_BEST.sort_values('Cover class').reset_index(drop=True)
CLASS_BEST['Cover class']=CLASS_BEST['Cover class'].astype(str)
assert CLASS_BEST.n.sum()==92
CLASS_BEST.to_csv(OUT/'table_06_RGB_MS_global_cover_classes.csv',index=False)
print('GLOBAL RGB+MS — 92 matched footprints; classes defined by observed Musanga cover')
display(CLASS_BEST.round(3))


## 02 — Figures 1–3: Nadine field comparison, RGB baseline, and multisensor cover accuracy

In [ ]:
plt.rcParams.update({'font.size':10,'axes.titlesize':12,'figure.dpi':110})
def save_fig(fig,name):
    p=FIG/name
    fig.savefig(p,dpi=210,bbox_inches='tight',facecolor='white')
    plt.show()
    print('Saved',p.name)
    return p
PALETTE={'Baego':'#3068a0','MOS':'#d18a2b',
         'Yangambi':'#3a9871','Yoko':'#9267ab'}
# Figure 1 — independent field basal area vs Nadine UAV crown fractions.
prim=NADINE.loc[NADINE.primary_BA_validation.astype(bool)].copy()
fig,axes=plt.subplots(1,2,figsize=(12,5),layout='constrained')
for ax,col,title in zip(axes,
        ('crown_fraction_plot_pct','musanga_fraction_digitized_crowns_pct'),
        ('Musanga crown area / plot area','Musanga area / all digitized crowns')):
    for site,q in prim.groupby('Site'):
        ax.scatter(q.field_BA_pct,q[col],s=38,alpha=.8,
                   color=PALETTE.get(site,'grey'),label=f'{site} (n={len(q)})')
    ax.plot([0,100],[0,100],ls='--',c='0.45',lw=1)
    ax.set(xlim=(0,100),ylim=(0,100),title=title,
           xlabel='Field Musanga basal-area fraction (%)',
           ylabel='Nadine UAV Musanga crown-area fraction (%)')
    ax.grid(alpha=.18)
axes[0].legend(frameon=False,fontsize=8)
fig.suptitle('Independent structural-dominance comparison (n=77 primary plots)',fontsize=13)
FIG1=save_fig(fig,'Fig01_field_basal_area_vs_Nadine_UAV_crowns.png')
# Figure 2 — full 106-footprint RGB baseline, with 4 biggest cover errors highlighted.
top4=RGB.nlargest(4,'abs_cover_error_pp')
fig,ax=plt.subplots(figsize=(7,6.3),layout='constrained')
for site,q in RGB.groupby('site'):
    ax.scatter(q.observed_cover_pp,q.predicted_cover_pp,
        color=PALETTE.get(site,'gray'),s=38,alpha=.75,label=f'{site} (n={len(q)})')
ax.plot([0,100],[0,100],'--',c='0.4',lw=1)
for q in top4.itertuples():
    ax.scatter([q.observed_cover_pp],[q.predicted_cover_pp],s=135,
        linewidth=1.5,facecolors='none',edgecolors='crimson')
    ax.annotate(str(q.unit_id),(q.observed_cover_pp,q.predicted_cover_pp),
                xytext=(4,4),textcoords='offset points',fontsize=8)
allrow=ALL_RGB.loc[ALL_RGB.Site.eq('ALL')].iloc[0]
ax.text(.04,.96,f'R² = {allrow.R2:.3f}\nMAE = {allrow.MAE_pp:.2f} pp',
    transform=ax.transAxes,va='top',bbox=dict(fc='white',alpha=.85,ec='none'))
ax.set(xlim=(-2,102),ylim=(-2,102),
       xlabel='Reference Musanga cover (%)',
       ylabel='Predicted Musanga cover (%)',
       title='Frozen RGB Model A — all 106 held-out footprints')
ax.grid(alpha=.18)
ax.legend(frameon=False,fontsize=8,loc='lower right')
FIG2=save_fig(fig,'Fig02_RGB_full_106_heldout.png')
# Figure 3 — matched common support: all completed models.
N=len(models)
fig,axes=plt.subplots(1,N,figsize=(5*N,5),layout='constrained')
if N==1:axes=[axes]
order=[x for x in ['RGB','RGB+DSM','RGB+MS','RGB+DSM+MS'] if x in models]
for ax,model in zip(axes,order):
    g=MATCH[MATCH.model.eq(model)]
    for site,q in g.groupby('site'):
        ax.scatter(q.observed_cover_pp,q.predicted_cover_pp,
            color=PALETTE.get(site,'gray'),s=25,alpha=.7,label=site)
    ax.plot([0,100],[0,100],'--',c='0.5',lw=1)
    m=unit_metrics(g)
    ax.text(.04,.96,f'R²={m["R2"]:.3f}\nMAE={m["MAE_pp"]:.2f} pp',
            transform=ax.transAxes,va='top',fontsize=9,
            bbox=dict(fc='white',alpha=.9,ec='none'))
    ax.set(xlim=(-2,102),ylim=(-2,102),aspect='equal',title=model,
           xlabel='Observed cover (%)',ylabel='Predicted cover (%)')
    ax.grid(alpha=.16)
axes[0].legend(frameon=False,fontsize=8,loc='lower right')
fig.suptitle(f'Sensor-matched held-out fractional cover (n={nmatch} footprints)',fontsize=13)
FIG3=save_fig(fig,'Fig03_matched_model_comparison.png')


## 03 — Supplementary site-level and MOS high-cover statistics (CSV only)

Calculate and preserve site-specific and high-cover MOS results without filling the scientific report with additional tables or figures. These can be explored separately if needed.

In [ ]:
# 03 — Preserve detailed diagnostics in CSV, but keep NB06 report concise.
high=MATCH[(MATCH.site=='MOS')&(MATCH.observed_cover_pp>40)]
highs=pd.DataFrame([dict(Model=mod,**unit_metrics(g))
                    for mod,g in high.groupby('model')])
highs.to_csv(OUT/'table_04_MOS_high_cover.csv',index=False)
BOOT_FILE=NB05/'08_four_arm_group_bootstrap.csv'
if not BOOT_FILE.exists():BOOT_FILE=NB05/'06_spatial_group_bootstrap.csv'
if BOOT_FILE.exists():
    BOOT=pd.read_csv(BOOT_FILE)
    BOOT.to_csv(OUT/'table_05_bootstrap_uncertainty.csv',index=False)
else:BOOT=None
print('MOS and bootstrap supplementary data saved as CSV (not displayed).')


## 04 — RGB reference imagery and annotation footprints

Create **actual raster excerpts** for representative complete circles and Nadine's exhaustive plots (including a MOS high-cover case). Musanga crowns are overlaid **red**, other annotated crowns **cyan**, and reference footprints **yellow**. A circle may not have explicit negatives; do not infer absence of Musanga from unlabeled pixels outside exhaustive footprints.

The image panels use source annotations and high-resolution RGB orthomosaics; display is downsampled to 1000 px maximum per excerpt. No synthetic imagery.

In [ ]:
UNITS=gpd.read_file(SPLIT_ROOT/'01_reference_groups_and_folds_v2.gpkg').to_crs('EPSG:32635')
UNITS.unit_id=UNITS.unit_id.astype(str)
EVAL=UNITS.merge(OWNERS[['unit_type','unit_id','mosaic','roi_mosaic_coverage_fraction']],
    on=['unit_type','unit_id'],validate='one_to_one')
EVAL=EVAL[EVAL.roi_mosaic_coverage_fraction.ge(.95)].copy()
assert len(EVAL)==106
EVAL=EVAL.merge(RGB[['unit_type','unit_id','observed_cover_pp']],
    on=['unit_type','unit_id'],validate='one_to_one')
CROWN=gpd.read_file(F78['exhaustive_plot_crowns']).to_crs(EVAL.crs)
CROWN['musanga']=pd.to_numeric(CROWN.Class_ID,errors='coerce').eq(1)
CIRCLE_CROWN=gpd.read_file(F78['circle_positive_crowns']).to_crs(EVAL.crs)
CIRCLE_CROWN['musanga']=True
def repaired_poly(g):
    if g is None or g.is_empty:return None
    if g.is_valid:return g
    try:
        k=make_valid(g)
        parts=[q for q in getattr(k,'geoms',[k])
               if q.geom_type in ('Polygon','MultiPolygon')]
        return unary_union(parts) if parts else None
    except Exception:return None
CROWN.geometry=[repaired_poly(g) for g in CROWN.geometry]
CIRCLE_CROWN.geometry=[repaired_poly(g) for g in CIRCLE_CROWN.geometry]
CROWN=CROWN[CROWN.geometry.notna()].copy()
CIRCLE_CROWN=CIRCLE_CROWN[CIRCLE_CROWN.geometry.notna()].copy()
def selected_footprints():
    result=[]
    for site,typ in [('Baego','circle'),('MOS','circle'),
                     ('MOS','plot'),('Yangambi','plot'),('Yoko','plot')]:
        q=EVAL[(EVAL.site.eq(site))&(EVAL.unit_type.eq(typ))].copy()
        if not len(q):continue
        if site=='MOS':
            q=q.sort_values('observed_cover_pp',ascending=False)
        else:
            q=q.assign(delta=(q.observed_cover_pp-q.observed_cover_pp.median()).abs()
                      ).sort_values('delta')
        result.append(q.iloc[0])
    return result
SAMPLES=selected_footprints()
def figure_window(ds,g,margin_m=12,max_px=1000):
    # V4 reference geometries are UTM 32635; their imaging CRS is usually same.
    from rasterio.warp import transform_bounds
    if ds.crs.to_string()!='EPSG:32635':
        bounds=transform_bounds('EPSG:32635',ds.crs,*g.bounds,densify_pts=21)
    else:bounds=g.bounds
    px=max(abs(ds.transform.a),abs(ds.transform.e))
    pad=max(margin_m,10*px)
    w=from_bounds(bounds[0]-pad,bounds[1]-pad,
                  bounds[2]+pad,bounds[3]+pad,ds.transform)
    c0=max(0,int(np.floor(w.col_off)));r0=max(0,int(np.floor(w.row_off)))
    c1=min(ds.width,int(np.ceil(w.col_off+w.width)))
    r1=min(ds.height,int(np.ceil(w.row_off+w.height)))
    win=Window(c0,r0,max(1,c1-c0),max(1,r1-r0))
    scale=min(1,max_px/max(win.width,win.height))
    return win,max(1,int(win.height*scale)),max(1,int(win.width*scale))
def display_rgb(ds,win,h,w):
    a=ds.read([1,2,3],window=win,out_shape=(3,h,w),
              resampling=Resampling.average,masked=True)
    a=np.asarray(a.filled(0),np.float32)
    if ds.dtypes[0]=='uint8':
        a=np.clip(a/255.,0,1)
    else:
        p=np.percentile(a[:,np.isfinite(a).all(axis=0)],[2,98])
        a=np.clip((a-p[0])/max(p[1]-p[0],1e-6),0,1)
    return np.moveaxis(a,0,-1)
def bounds_extent(ds,win):
    b=rasterio.windows.bounds(win,ds.transform)
    return [b[0],b[2],b[1],b[3]]
def geom_to_ds(g,ds):
    return gpd.GeoSeries([g],crs='EPSG:32635').to_crs(ds.crs).iloc[0]
def paint_annotations(ax,rec,ds):
    src=CIRCLE_CROWN if rec.unit_type=='circle' else CROWN
    geom=rec.geometry
    # Spatial index before CRS transformation keeps this bounded.
    hits=src.iloc[src.sindex.query(geom,predicate='intersects')].copy()
    hits=hits[hits.geometry.notna()]
    if len(hits):
        hits=hits.to_crs(ds.crs)
        for good,rgba in ((False,'#38c2db'),(True,'#e64a43')):
            selected=hits[hits.musanga.eq(good)]
            if len(selected):
                selected.boundary.plot(ax=ax,color=rgba,linewidth=.8)
    border=geom_to_ds(geom,ds)
    gpd.GeoSeries([border],crs=ds.crs).boundary.plot(ax=ax,color='#fff04c',linewidth=2.0)
    return len(hits)
fig,axs=plt.subplots(len(SAMPLES),2,figsize=(10,4.0*len(SAMPLES)),layout='constrained')
ANNOTATION_ROWS=[]
for i,rec in enumerate(SAMPLES):
    ds_path=MOSAICS[str(rec.mosaic)]
    with rasterio.open(ds_path) as ds:
        win,h,w=figure_window(ds,rec.geometry)
        rgb=display_rgb(ds,win,h,w)
        extent=bounds_extent(ds,win)
        for ax in axs[i]:
            ax.imshow(rgb,extent=extent,origin='upper',interpolation='nearest')
            ax.set_xlim(extent[0],extent[1]);ax.set_ylim(extent[2],extent[3])
            ax.tick_params(labelsize=7)
            ax.set_aspect('equal')
        n_ann=paint_annotations(axs[i,1],rec,ds)
        axs[i,0].set_title(f'{rec.site} | {rec.unit_type} {rec.unit_id} | RGB')
        axs[i,1].set_title(f'Annotated crown polygons | {n_ann} intersecting')
        ANNOTATION_ROWS.append(dict(site=rec.site,type=rec.unit_type,
                         unit_id=str(rec.unit_id),observed_cover_pp=rec.observed_cover_pp,
                         n_annotation_candidates=n_ann,mosaic=rec.mosaic))
fig.suptitle('Actual UAV reference footprints: RGB versus crown annotations',fontsize=14)
FIG6=save_fig(fig,'Fig06_reference_RGB_and_annotations.png')
pd.DataFrame(ANNOTATION_ROWS).to_csv(OUT/'figure_06_reference_samples.csv',index=False)


## 05 — Actual DSM and MS panels for the same reference footprints

Visualize registered **DSM relative surface elevation**, **NIR**, **red edge**, and an **MS false-colour composite (NIR/red-edge/red)** next to RGB and the original reference boundary. These views are *interpretive illustrations* of sensor information, not model predictions. The DSM elevation shown is the original surface elevation normalized by its local median—not any single multiscale-feature channel. All reads are limited to the selected reference windows.

In [ ]:
PAIRS=pd.read_csv(INPUT00/'00_rgb_dsm_pair_audit.csv')
DSM_LOOKUP={Path(z.rgb_path).name:Path(z.dsm_path) for z in PAIRS.itertuples(index=False)}
assert set(MOSAICS).issubset(DSM_LOOKUP)
REG=pd.read_csv(INPUT00/'00_ms_raster_headers.csv')
REG=REG[REG.error.fillna('').eq('')].copy()
SITE_MS_GROUPS={
    'Baego.tif':['Baego_MS'],
    'Mos_1_2_transparent_mosaic_group1_georef.tif':['MOS_1_2_MS'],
    'Mos_3_4_transparent_mosaic_group1_georef.tif':['MOS_3_4_MS'],
    'Mos_5_6_transparent_mosaic_group1_georef.tif':['MOS_5_6_MS'],
    'Yangambi_georef.tif':['Yangambi_YBJoseph_right_MS','Yangambi_YBForgeo_right_MS']}
def ms_registry(mosaic):
    groups=SITE_MS_GROUPS.get(mosaic,[])
    allsets=[]
    for group in groups:
        part=REG[REG.group.eq(group)]
        if group=='Baego_MS':
            if len(part)==1:
                path=Path(part.iloc[0].path)
                if path.is_file(): allsets.append((group,{b:(path,i+1)
                    for i,b in enumerate(('red','green','red_edge','nir'))}))
        else:
            bands={}
            for band in ('red','green','red_edge','nir'):
                hits=part[part.band_hint.eq(band)]
                if len(hits)>1 and group=='MOS_1_2_MS' and band=='red_edge':
                    hits=hits[hits.name.eq('mlt_Mos_1_2_transparent_mosaic_red edge_georef.tif')]
                if len(hits)==1:
                    p=Path(hits.iloc[0].path)
                    if p.is_file():bands[band]=(p,1)
            if len(bands)==4:allsets.append((group,bands))
    return allsets
def robust_stretch(a,valid):
    z=np.asarray(a,float)
    good=np.isfinite(z)&valid
    if not good.any():return np.zeros_like(z,dtype='float32')
    lo,hi=np.percentile(z[good],[2,98])
    return np.clip((np.where(good,z,lo)-lo)/max(hi-lo,1e-6),0,1).astype('float32')
def aligned_band(path,band,ds,win,h,w):
    transform=ds.window_transform(win)*Affine.scale(win.width/w,win.height/h)
    with rasterio.open(path) as src:
        with WarpedVRT(src,crs=ds.crs,transform=transform,
                       width=w,height=h,resampling=Resampling.bilinear) as vrt:
            arr=vrt.read(band,masked=True,out_dtype='float32')
            data=np.asarray(arr.filled(np.nan),np.float32)
            ok=~np.ma.getmaskarray(arr)&np.isfinite(data)
    return data,ok
# Select only reference footprints with REAL, spatially overlapping MS data.
# Do not confuse a registered MS flight with coverage over a particular ROI.
# A blank MS tile is unacceptable in the scientific report.
MIN_MS_FOOTPRINT_FRACTION=.75
MIN_DSM_FOOTPRINT_FRACTION=.75
MAX_DISPLAY_PX=500
MAX_TRIES_PER_SITE=18
site_order=('Baego','MOS','Yangambi')
example_rows=[]
omitted=[]
for site in site_order:
    eligible=EVAL[EVAL.site.eq(site)].copy()
    # Restrict to the exact 92 eligible model-comparison footprints.
    paired=MATCH[MATCH.model.eq('RGB')][['unit_type','unit_id']].drop_duplicates()
    eligible=eligible.merge(paired,on=['unit_type','unit_id'],how='inner')
    # At MOS, prefer dense Musanga, but skip footprints with incomplete MS coverage.
    eligible=eligible.sort_values('observed_cover_pp',
                  ascending=(site!='MOS'))
    chosen=None
    for _,rec in eligible.head(MAX_TRIES_PER_SITE).iterrows():
        mosaic=str(rec.mosaic)
        groups=ms_registry(mosaic)
        if not groups:
            omitted.append(dict(site=site,unit_id=str(rec.unit_id),
                                cause='No registered 4-band MS flight'))
            continue
        try:
            with rasterio.open(MOSAICS[mosaic]) as ds:
                win,h,w=figure_window(ds,rec.geometry,
                                margin_m=8,max_px=MAX_DISPLAY_PX)
                if h<10 or w<10:continue
                extent=bounds_extent(ds,win)
                tr=ds.window_transform(win)*Affine.scale(win.width/w,win.height/h)
                footprint=rasterize([(geom_to_ds(rec.geometry,ds),1)],
                            out_shape=(h,w),transform=tr,
                            fill=0,dtype='uint8').astype(bool)
                if int(footprint.sum())<150:continue
                dem,okdem=aligned_band(DSM_LOOKUP[mosaic],1,ds,win,h,w)
                dsmpct=float(np.mean(okdem[footprint]))
                if dsmpct<MIN_DSM_FOOTPRINT_FRACTION:
                    omitted.append(dict(site=site,unit_id=str(rec.unit_id),
                                        cause=f'DSM support {dsmpct:.1%}'))
                    continue
                candidates=[]
                for group,bands in groups:
                    layers={b:aligned_band(path,idx,ds,win,h,w)
                            for b,(path,idx) in bands.items()}
                    valid=np.logical_and.reduce([
                        layers[b][1] & np.isfinite(layers[b][0])
                        & (layers[b][0]>0)
                        for b in ('red','green','red_edge','nir')])
                    fraction=float(np.mean(valid[footprint]))
                    candidates.append((fraction,group,layers,valid))
                coverage,flight,layers,valid=max(candidates,key=lambda z:z[0])
                if coverage<MIN_MS_FOOTPRINT_FRACTION:
                    omitted.append(dict(site=site,unit_id=str(rec.unit_id),
                                       cause=f'MS footprint coverage {coverage:.1%}'))
                    continue
                rgb=display_rgb(ds,win,h,w)
                if not np.isfinite(rgb).all():continue
                chosen=dict(site=site,rec=rec,mosaic=mosaic,
                    rgb=rgb,dem=dem,okdem=okdem,extent=extent,
                    flight=flight,layers=layers,valid=valid,
                    ms_coverage=coverage,dsm_coverage=dsmpct,
                    shape=(h,w))
                break
        except Exception as err:
            omitted.append(dict(site=site,unit_id=str(rec.unit_id),
                                cause=f'Read failed: {str(err)[:100]}'))
    if chosen:
        example_rows.append(chosen)
    else:
        print('NO COVERAGE-COMPLETE EXAMPLE at',site,
              '— omitting it rather than plotting empty MS panels.')
assert len(example_rows)>=1,(
    'No examples with sufficiently complete MS and DSM coverage; '
    'audit MS georeferencing/registry before generating this figure')
fig,axs=plt.subplots(len(example_rows),4,
                     figsize=(14,3.6*len(example_rows)),squeeze=False,
                     layout='constrained')
MS_EXAMPLES=[]
for i,case in enumerate(example_rows):
    rec=case['rec'];rgb=case['rgb'];extent=case['extent']
    dem=case['dem'];okdem=case['okdem']
    rel=np.where(okdem,dem-np.median(dem[okdem]),np.nan)
    layers=case['layers'];valid=case['valid']
    rr,re,nir=(layers[k][0] for k in ('red','red_edge','nir'))
    false=np.stack([robust_stretch(nir,valid),
                    robust_stretch(re,valid),
                    robust_stretch(rr,valid)],axis=-1)
    # Missing source pixels remain white, not silently imputed or treated
    # as zeros; reference footprints were selected for >=75% valid MS.
    false[~valid]=1.
    panels=[rgb,rel,false,valid.astype(float)]
    titles=['RGB orthomosaic','DSM relative elevation (m)',
            'MS false colour (NIR / RE / red)','4-band MS validity mask']
    with rasterio.open(MOSAICS[case['mosaic']]) as ds:
        border=geom_to_ds(rec.geometry,ds)
        for j,(ax,img,title) in enumerate(zip(axs[i],panels,titles)):
            if j==1:ax.imshow(img,extent=extent,origin='upper',cmap='terrain')
            elif j==3:ax.imshow(img,extent=extent,origin='upper',cmap='Greys',vmin=0,vmax=1)
            else:ax.imshow(img,extent=extent,origin='upper')
            gpd.GeoSeries([border],crs=ds.crs).boundary.plot(
                ax=ax,color='#fff04c',linewidth=1.5)
            ax.set_xlim(extent[0],extent[1]);ax.set_ylim(extent[2],extent[3])
            ax.set_xticks([]);ax.set_yticks([])
            if i==0:ax.set_title(title)
        axs[i,0].set_ylabel(
            f"{case['site']}: {rec.unit_type} {rec.unit_id}",fontsize=9)
    MS_EXAMPLES.append(dict(site=case['site'],
                 unit_type=rec.unit_type,unit_id=str(rec.unit_id),
                 mosaic=case['mosaic'],ms_flight=case['flight'],
                 ms_footprint_fraction=case['ms_coverage'],
                 dsm_footprint_fraction=case['dsm_coverage']))
fig.suptitle('Representative source imagery: only footprints with verified MS coverage')
FIG7=save_fig(fig,'Fig07_RGB_DSM_MS_same_footprints.png')
pd.DataFrame(MS_EXAMPLES).to_csv(OUT/'figure_07_sensor_examples_QA.csv',index=False)
pd.DataFrame(omitted).to_csv(OUT/'figure_07_excluded_missing_sensor_coverage.csv',index=False)
display(pd.DataFrame(MS_EXAMPLES))
print('Skipped',len(omitted),'candidate footprints with missing/poor MS or DSM coverage.')


## 06 — Assemble the scientific Word report with actual figures and tables

In [ ]:
from docx import Document
from docx.shared import Inches,Pt,RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from datetime import date
DOC=Document()
sec=DOC.sections[0]
sec.top_margin=Inches(.7);sec.bottom_margin=Inches(.65)
sec.left_margin=Inches(.75);sec.right_margin=Inches(.75)
styles=DOC.styles
styles['Normal'].font.name='Aptos'
styles['Normal'].font.size=Pt(9)
styles['Normal'].paragraph_format.space_after=Pt(5)
for key,size in [('Title',20),('Heading 1',14),('Heading 2',11)]:
    styles[key].font.name='Aptos'
    styles[key].font.size=Pt(size)
    styles[key].font.color.rgb=RGBColor(30,65,87)
def paragraph(text):
    DOC.add_paragraph(text)
def heading(s,level=1):DOC.add_heading(s,level)
def add_table(frame,cols=None,dec=3):
    if cols is not None:frame=frame[cols]
    t=DOC.add_table(rows=1,cols=len(frame.columns))
    t.style='Light Shading Accent 1'
    t.alignment=WD_TABLE_ALIGNMENT.CENTER
    for i,k in enumerate(frame.columns):t.rows[0].cells[i].text=str(k)
    for row in frame.itertuples(index=False,name=None):
        cells=t.add_row().cells
        for i,v in enumerate(row):
            cells[i].text=f'{v:.{dec}f}' if isinstance(v,(float,np.floating)) and np.isfinite(v) else str(v)
    DOC.add_paragraph('')
def figure(path,caption):
    assert Path(path).is_file(),path
    max_width=6.95
    DOC.add_picture(str(path),width=Inches(max_width))
    p=DOC.add_paragraph(caption)
    p.style='Caption'
DOC.add_heading('UAV mapping of Musanga: RGB, multiscale DSM and multispectral information',0)
paragraph('Internal scientific progress report | Musanga V4 | 19 November 2026')
paragraph('Prepared from validated GIS reference data and spatially held-out U-Net evaluation outputs; all source images and numerical tables are generated by NB06.')
heading('Executive summary')
row106=ALL_RGB[ALL_RGB.Site=='ALL'].iloc[0]
mrgb=MATCH_SUM[(MATCH_SUM.Site=='ALL')&(MATCH_SUM.Model=='RGB')].iloc[0]
ms=MATCH_SUM[(MATCH_SUM.Site=='ALL')&(MATCH_SUM.Model=='RGB+MS')].iloc[0]
paragraph(f'The frozen RGB-only ResNet18 U-Net achieved R²={row106.R2:.3f}, MAE={row106.MAE_pp:.2f} percentage points and RMSE={row106.RMSE_pp:.2f} pp over 106 spatially held-out reference footprints. On the 92-footprint common RGB/DSM/MS support, RGB had R²={mrgb.R2:.3f}; RGB+MS achieved R²={ms.R2:.3f} and RMSE={ms.RMSE_pp:.2f} pp. Gains were concentrated at MOS, while the additional modalities increased low-cover overprediction at Yangambi. Because the baseline is frozen and fusion models were further fine-tuned, additional training is a possible contributor to the improvements.')
heading('Key results')
paragraph('The complete RGB benchmark includes 106 held-out footprints. The strict multisensor comparison includes 92 footprints with valid RGB/DSM/MS support. Each footprint has equal weight; 14 incomplete-sensor footprints are excluded, not scored as zero.')
heading('Full RGB baseline (106 footprints)',2)
add_table(FULL_RGB[['Site','n','R2','MAE_pp','RMSE_pp','F1']])
heading('Comparison of four models: overall (92 matched footprints)',2)
add_table(SUMMARY_MAIN[SUMMARY_MAIN.Site.eq('ALL')][
    ['Model','n','R2','MAE_pp','RMSE_pp','Bias_pp','F1','IoU']])
heading('Best model RGB+MS: error across observed-cover classes',2)
add_table(CLASS_BEST[['Cover class','n','F1','IoU','MAE (pp)','RMSE (pp)','Bias (pp)']])
paragraph('The class table includes only the global RGB+MS model on the matched 92-footprint cohort. R² is intentionally omitted within narrow cover classes because low observed variance makes it unstable.')
heading('1. Research objectives')
paragraph('The study evaluates whether adding UAV-derived surface structure (multiscale DSM) and multispectral information to RGB improves pixel-level Musanga crown segmentation and fractional canopy-cover estimation. Specific questions concern the incremental contribution of structure versus spectrum, prediction at high cover at MOS, low-cover false positives at Yangambi, and generalization across three spatially blocked held-out folds.')
heading('2. Study sites and reference datasets')
paragraph('Four Congolese forest landscapes were sampled: Baego, MOS, Yangambi and Yoko. The evaluation dataset comprises 30 complete annotated circles and 76 exhaustively annotated field plots. They are treated as 106 reference footprints, with exactly equal statistical weighting regardless of geometry or footprint size. The two source types differ in reference mapping history but not in their weight in the final score.')
paragraph('The NB00 field dataset contains 78 exhaustively delineated plots, with 77 primary plots in the independent field basal-area comparison and one Yoko partial-census geometry retained as a sensitivity case. Field percentage of Musanga basal area and UAV Musanga crown-area cover are distinct quantities and should not be interpreted as interchangeable ground-truth measurements.')
figure(FIG6,'Figure 1. Original RGB imagery and genuine mapped reference crowns. Red boundaries indicate Musanga crown annotations, cyan boundaries other annotated crowns, and yellow boundaries the validation footprint.')
figure(FIG1,'Figure 2. Independent field Musanga basal-area fraction versus Nadine UAV crown-area fractions (77 primary field plots). This is a comparison of different structural dominance measures, not prediction model accuracy.')
heading('3. UAV predictors and data processing')
paragraph('The RGB baseline comprises three-channel UAV orthomosaics. The 13-channel RGB+DSM model adds ten structure predictors: local prominence, local surface roughness and convexity, plus annular relative-surface features (5–15, 15–30, 30–60 m), neighborhood surface standard deviations at 15, 30 and 60 m, and contextual valid-data fraction. Contextual DSM features are computed from a spatially anchored 1-m elevation surface with shared contextual calculations, with sampled checks against the historical exact per-chip implementation.')
paragraph('Four-band MS information comprises red, green, red-edge and near infrared. Bands are aligned to RGB and expressed as within-acquisition relative spectra. Four-band acquisitions are never merged band-by-band across separate flights. MS absence is recorded as missing support rather than zero Musanga.')
figure(FIG7,'Figure 3. RGB, DSM and multispectral false-colour views from reference footprints selected for verified four-band MS coverage (at least 75% of each footprint). The validity mask explicitly shows missing MS pixels; excluded examples are logged in the figure QA table.')
heading('4. Model training and spatial validation')
paragraph('The RGB baseline (Model A) is a ResNet18 U-Net trained on exhaustive circles with supplementary mapped crowns, and kept frozen in the evaluation. RGB+DSM and RGB+MS U-Nets were initialized from its three-channel parameters with zero-valued extra input weights and fine-tuned on the same 1,392 jointly valid cached chips (500/463/429 chips across three outer-fold training caches). The optional 17-channel RGB+DSM+MS arm is included only when its held-out results exist.')
paragraph('The original complete RGB evaluation uses 106 independent held-out footprints. Sensor-matched analysis is based on 92 footprints with at least some common valid RGB, DSM and complete MS coverage; the other 14 are transparently documented as unavailable. Every reported modality score uses identical supervised pixels within each matched footprint. Cover metrics are computed from one predicted/observed cover pair per footprint; F1, IoU, precision, recall and Brier are the unweighted averages of per-footprint pixel scores.')
heading('5. Results: complete RGB baseline')
figure(FIG2,'Figure 4. Frozen RGB Model A on all 106 reference footprints. The four largest errors are outlined.')
heading('6. Results: strictly sensor-matched comparison')
figure(FIG3,f'Figure 5. Observed versus predicted cover for four models on {nmatch} matched reference footprints.')
heading('7. Interpretation: high-cover MOS and Yangambi')
paragraph('At MOS, additional spectral information strongly reduces the underprediction seen in the RGB baseline, particularly at high cover. At Yangambi, where observed cover variance is small, added features increase positive prediction bias and cause strongly negative R² values. Consequently, improvements in study-wide cover RMSE coexist with small or negative changes in macro-averaged segmentation F1.')
heading('8. Different MOS R² values reflect different evaluation samples')
paragraph('MOS RGB R²=0.164 in NB04 is calculated over 29 complete RGB-supported footprints. The NB05 matched-sensor RGB R²=−0.025 uses 23 footprints and only their jointly RGB/DSM/MS-valid pixels. The observed-cover variance and sample composition therefore differ, although the RGB model itself is identical. An earlier MOS R²≈0.186 used an additional 50/50 circle/plot reweighting; that weighting has been superseded by equal-footprint reporting.')
heading('9. Uncertainty and limitations')
paragraph('The paired spatial-group bootstrap assesses differences in mean footprint absolute cover error while resampling spatial groups rather than individual pixels. Confidence intervals that include zero do not support strong claims of general study-wide improvement. The three outer folds are spatial holdouts, not whole-forest site holdouts. The low-variance Yangambi cohort makes R² unstable as a measure of ecological usefulness.')
paragraph('Additional methodological limitations: incomplete MS coverage (14/106 unavailable); remaining uncertainties about crown-scale registration; unequal model optimization (frozen RGB versus further fine-tuned fusion arms); provisional exhaustive-background semantics; and local observations at MOS with low-stature Musanga in cropland. None of the four severe MOS cases were removed from the original full-RGB evaluation.')
heading('10. Recommended next analyses')
paragraph('Train a same-support RGB-only fine-tuning control, examine MOS high-cover failures and Yangambi false positives, and test transfer to independent forests. RGB+MS is the current preferred modality for mapping, although sensor-specific gains remain site dependent.')
report=OUT/'Musanga_UAV_Multisensor_Scientific_Progress_Report.docx'
DOC.save(report)
print('SCIENTIFIC WORD REPORT CREATED:',report)
print('Figures:',len(list(FIG.glob('Fig*.png'))))
print('Tables:',len(list(OUT.glob('table_*.csv'))))


## 07 — Final output checklist and downloadable report

In [ ]:
from IPython.display import FileLink
required=[
 OUT/'Musanga_UAV_Multisensor_Scientific_Progress_Report.docx',
 *[FIG/f'Fig{z:02d}_{name}.png' for z,name in [
  (1,'field_basal_area_vs_Nadine_UAV_crowns'),
  (2,'RGB_full_106_heldout'),
  (3,'matched_model_comparison'),
  (6,'reference_RGB_and_annotations'),
  (7,'RGB_DSM_MS_same_footprints')]]
]
assert all(p.is_file() and p.stat().st_size>1000 for p in required),[
    p for p in required if not p.exists()]
manifest=dict(notebook='NB06_v4_004',generated_at='2026-11-19',
    original_units=106,matched_units=92,model_arms=models,
    primary_metric_weighting='equal reference footprints',
    sources={'NB04':str(BASE_CSV),'NB05':str(FOUR_PATH if FOUR_PATH.exists() else METRIC_PATH),
             'Nadine':str(NADINE_CSV),'DSM_pairs':str(INPUT00/'00_rgb_dsm_pair_audit.csv')},
    files=[str(p) for p in required])
(OUT/'06_report_manifest.json').write_text(json.dumps(manifest,indent=2))
print('All scientific figures and tables present.')
display(FileLink(str(required[0])))
print('Report folder:',OUT)
